# 🛰️ SatQuery AI — GeoChat-7B Kaggle GPU Backend

Run **GeoChat-7B** (MBZUAI/geochat-7b) on a free Kaggle GPU (Tesla T4 x 2 or P100) with FastAPI & Cloudflare tunnel.

### ⚡ Quick Start:
1. In the right sidebar: Set **Accelerator** to `GPU T4 x 2` or `GPU P100`.
2. Set **Internet** to `Internet on` (required for downloading packages and model weights).
3. Click **Run All**.
4. Copy the generated `GEOCHAT_API_URL` into your SatQuery AI `.env`!

In [ ]:
# CELL 1 — GPU CHECK
import sys
import torch
import subprocess

print("=" * 60)
print("SATQUERY AI — GEOCHAT GPU BACKEND")
print("=" * 60)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Enable Kaggle GPU (Tesla T4).")

for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    print(f"GPU {i}: {name} | {mem_gb:.2f} GB VRAM")

print("\n✅ GPU READY")


In [ ]:
# CELL 2 — DEPENDENCIES
import os
import sys
import subprocess

GE0CHAT_DIR = "/kaggle/working/GeoChat"

packages = [
    "einops==0.7.0",
    "einops-exts==0.0.4",
    "sentencepiece",
    "peft==0.11.1",
    "timm==0.9.12",
    "shortuuid",
    "bitsandbytes>=0.41.0",
    "accelerate==0.25.0",
    "markdown2",
    "scikit-learn",
    "fastapi",
    "uvicorn[standard]",
    "python-multipart",
    "requests",
    "Pillow",
]

print("Installing required dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True,
)

import transformers
import accelerate
import bitsandbytes

print(f"📦 PyTorch: {torch.__version__}")
print(f"📦 Transformers: {transformers.__version__}")
print(f"📦 Accelerate: {accelerate.__version__}")
print(f"📦 BitsAndBytes: {bitsandbytes.__version__}")
print("✅ Dependencies installed successfully")


In [ ]:
# CELL 3 — CLONE GEOCHAT
import os
import sys
import subprocess

GE0CHAT_DIR = "/kaggle/working/GeoChat"

if not os.path.exists(GE0CHAT_DIR):
    print("Cloning GeoChat repository...")
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/mbzuai-oryx/GeoChat.git",
            GE0CHAT_DIR,
        ],
        check=True,
    )

if GE0CHAT_DIR not in sys.path:
    sys.path.insert(0, GE0CHAT_DIR)

print("✅ GeoChat cloned into:", GE0CHAT_DIR)


In [ ]:
# CELL 4 — APPLY KAGGLE GEOCHAT FIX (CLIP DELAYED-LOADING)
import os
import re

CLIP_ENCODER_FILE = "/kaggle/working/GeoChat/geochat/model/multimodal_encoder/clip_encoder.py"

if not os.path.exists(CLIP_ENCODER_FILE):
    raise FileNotFoundError(f"Cannot find {CLIP_ENCODER_FILE}")

with open(CLIP_ENCODER_FILE, "r") as f:
    content = f.read()

# Fix: When delay_load=True, CLIPVisionTower must NOT instantiate CLIPVisionModel inside __init__
old_init = """        if not delay_load:
            self.load_model()
        else:
            self.cfg_only = CLIPVisionConfig.from_pretrained(self.vision_tower_name)
            self.image_processor = CLIPImageProcessor.from_pretrained(self.vision_tower_name)
            self.vision_tower = CLIPVisionModel.from_pretrained(self.vision_tower_name)
            self.vision_tower.requires_grad_(False)
            self.clip_interpolate_embeddings(image_size=504, patch_size=14)"""

new_init = """        if not delay_load:
            self.load_model()
        else:
            self.cfg_only = CLIPVisionConfig.from_pretrained(self.vision_tower_name)"""

if old_init in content:
    content = content.replace(old_init, new_init)
    with open(CLIP_ENCODER_FILE, "w") as f:
        f.write(content)
    print("✅ Applied exact CLIP delay_load patch to clip_encoder.py")
else:
    # Regex fallback if whitespace differs
    pattern = r"(\s+if not delay_load:\s+self\.load_model\(\)\s+else:)([\s\S]*?)(def load_model)"
    match = re.search(pattern, content)
    if match and "CLIPVisionModel.from_pretrained" in match.group(2):
        replacement = r"\1\n            self.cfg_only = CLIPVisionConfig.from_pretrained(self.vision_tower_name)\n\n    \3"
        content = re.sub(pattern, replacement, content)
        with open(CLIP_ENCODER_FILE, "w") as f:
            f.write(content)
        print("✅ Applied regex CLIP delay_load patch to clip_encoder.py")
    else:
        print("ℹ️ clip_encoder.py already patched or correct")

# Verify patch
with open(CLIP_ENCODER_FILE, "r") as f:
    verify_content = f.read()

init_section = verify_content.split("def load_model")[0]
if "self.vision_tower = CLIPVisionModel.from_pretrained" in init_section:
    raise RuntimeError("Verification failed: CLIPVisionModel is still instantiated inside __init__!")

print("✅ CLIP delayed-loading patch verified")


In [ ]:
# CELL 5 — OTHER COMPATIBILITY FIXES
import os
import sys
import torch

# 1. MPT prefix LM compatibility
MPT_FILE = "/kaggle/working/GeoChat/geochat/model/language_model/mpt/hf_prefixlm_converter.py"
if os.path.exists(MPT_FILE):
    with open(MPT_FILE, "r") as f:
        mpt_content = f.read()

    local_impl = '''
def _expand_mask_bloom(mask: torch.Tensor, dtype: torch.dtype, tgt_len: int = None):
    bsz, src_len = mask.size()
    tgt_len = tgt_len if tgt_len is not None else src_len
    expanded_mask = mask[:, None, None, :].expand(bsz, 1, tgt_len, src_len).to(dtype)
    inverted_mask = 1.0 - expanded_mask
    return inverted_mask.masked_fill(inverted_mask.to(torch.bool), torch.finfo(dtype).min)
'''
    if "def _expand_mask_bloom" not in mpt_content:
        mpt_content = mpt_content.replace("import torch\n", "import torch\n" + local_impl, 1)
        with open(MPT_FILE, "w") as f:
            f.write(mpt_content)
        print("✅ MPT compatibility patch applied")

# 2. DynamicCache compatibility
ARCH_FILE = "/kaggle/working/GeoChat/geochat/model/geochat_arch.py"
if os.path.exists(ARCH_FILE):
    with open(ARCH_FILE, "r") as f:
        arch_content = f.read()

    old_mask = """attention_mask = torch.ones((attention_mask.shape[0], past_key_values[-1][-1].shape[-2] + 1), dtype=attention_mask.dtype, device=attention_mask.device)"""
    new_mask = """# DynamicCache compatibility
                from transformers.cache_utils import DynamicCache

                if isinstance(past_key_values, DynamicCache):
                    past_len = past_key_values.get_seq_length()
                else:
                    past_len = past_key_values[-1][-1].shape[-2]

                attention_mask = torch.ones(
                    (attention_mask.shape[0], past_len + 1),
                    dtype=attention_mask.dtype,
                    device=attention_mask.device,
                )"""

    if old_mask in arch_content:
        arch_content = arch_content.replace(old_mask, new_mask)
        with open(ARCH_FILE, "w") as f:
            f.write(arch_content)
        print("✅ DynamicCache patch applied")

# 3. PreTrainedModel expert implementation safety
from transformers.modeling_utils import PreTrainedModel

@classmethod
def _can_set_experts_implementation_patched(cls):
    try:
        sys.modules[cls.__module__].__file__
        return True
    except KeyError:
        return False

PreTrainedModel._can_set_experts_implementation = _can_set_experts_implementation_patched

print("✅ All compatibility patches verified")


In [ ]:
# CELL 6 — DOWNLOAD MODEL
import os
from huggingface_hub import snapshot_download

MODEL_PATH = "/kaggle/working/geochat-7B"

if not os.path.exists(MODEL_PATH):
    print("Downloading MBZUAI/geochat-7B...")
    snapshot_download(
        repo_id="MBZUAI/geochat-7B",
        local_dir=MODEL_PATH,
        ignore_patterns=[
            "*.msgpack",
            "*.h5",
            "flax_model*",
        ],
    )

assert os.path.exists(os.path.join(MODEL_PATH, "config.json")), "config.json missing!"
print("✅ Model available at:", MODEL_PATH)


In [ ]:
# CELL 7 — LOAD GEOCHAT-7B
import os
import sys
import gc
import torch

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

GE0CHAT_DIR = "/kaggle/working/GeoChat"
if GE0CHAT_DIR not in sys.path:
    sys.path.insert(0, GE0CHAT_DIR)

from transformers import AutoTokenizer, BitsAndBytesConfig
from geochat.model import GeoChatLlamaForCausalLM
from geochat.constants import (
    DEFAULT_IMAGE_PATCH_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IM_END_TOKEN,
)

MODEL_PATH = "/kaggle/working/geochat-7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    use_fast=False,
)

print("Loading GeoChat-7B in 4-bit on CUDA...")
model = GeoChatLlamaForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
)

print("✅ LLaMA backbone loaded successfully")

# Resize token embeddings for multimodal special tokens
if getattr(model.config, "mm_use_im_patch_token", True):
    tokenizer.add_tokens([DEFAULT_IMAGE_PATCH_TOKEN], special_tokens=True)

if getattr(model.config, "mm_use_im_start_end", False):
    tokenizer.add_tokens([DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN], special_tokens=True)

model.resize_token_embeddings(len(tokenizer))


In [ ]:
# CELL 8 — LOAD VISION TOWER
print("Loading vision tower outside meta-device context...")

vision_tower = model.get_vision_tower()

if not vision_tower.is_loaded:
    vision_tower.load_model()

vision_tower.to(
    device="cuda",
    dtype=torch.float16,
)

image_processor = vision_tower.image_processor

assert vision_tower.is_loaded is True, "Vision tower failed to report is_loaded == True"
print("✅ Vision tower loaded and initialized in FP16 on CUDA")


In [ ]:
# CELL 9 — MODEL VALIDATION
free_gb, total_gb = torch.cuda.mem_get_info()
context_len = getattr(model.config, "max_sequence_length", 2048)

print("=" * 60)
print("MODEL VALIDATION REPORT")
print("=" * 60)
print("GeoChat model loaded: PASS")
print("LLaMA weights: PASS")
print("4-bit quantization: PASS")
print("Vision tower: PASS")
print("Vision weights: PASS")
print("Projector: PASS")
print("CUDA: PASS")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM Used: {(total_gb - free_gb) / 1e9:.2f} GB / {total_gb / 1e9:.2f} GB")
print(f"Context Length: {context_len}")
print("=" * 60)
print("✅ GEOCHAT-7B READY")


In [ ]:
# CELL 10 — REAL IMAGE INFERENCE
import os
from PIL import Image
import torch

from geochat.conversation import conv_templates, SeparatorStyle
from geochat.mm_utils import tokenizer_image_token, KeywordsStoppingCriteria
from geochat.constants import IMAGE_TOKEN_INDEX

# Ensure image processor dimensions
image_processor.size = {"height": 504, "width": 504}
image_processor.crop_size = {"height": 504, "width": 504}

def geochat_infer(image_input, question):
    if isinstance(image_input, str):
        image = Image.open(image_input).convert("RGB")
    else:
        image = image_input.convert("RGB")

    conv = conv_templates["vicuna_v1"].copy()
    conv.append_message(conv.roles[0], "<image>\n" + question)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    input_ids = tokenizer_image_token(
        prompt,
        tokenizer,
        IMAGE_TOKEN_INDEX,
        return_tensors="pt",
    ).unsqueeze(0).cuda()

    image_tensor = image_processor.preprocess(image, return_tensors="pt")["pixel_values"]
    if image_tensor.ndim == 3:
        image_tensor = image_tensor.unsqueeze(0)
    image_tensor = image_tensor.half().cuda()

    stop_str = conv.sep2 if conv.sep_style == SeparatorStyle.TWO else conv.sep
    stopping_criteria = KeywordsStoppingCriteria([stop_str], tokenizer, input_ids)

    model.config.use_cache = False

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=image_tensor,
            do_sample=False,
            max_new_tokens=256,
            stopping_criteria=[stopping_criteria],
            use_cache=False,
        )

    answer = tokenizer.decode(
        output_ids[0, input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    answer = answer.replace("▁", " ")
    answer = " ".join(answer.split())

    if answer.endswith(stop_str):
        answer = answer[:-len(stop_str)].strip()

    return answer

# Test with real satellite image
IMAGE_PATH = "/kaggle/input/datasets/sairam0564/test-geochat/20260723_ColoradoRiver_Blythe.png"
QUESTION = "Describe what you see in this satellite image."

if not os.path.exists(IMAGE_PATH):
    print(f"Notice: {IMAGE_PATH} not found directly, creating sample test image for validation...")
    test_img = Image.new("RGB", (504, 504), color=(34, 139, 34))
    test_img.save("/kaggle/working/test_satellite.png")
    IMAGE_PATH = "/kaggle/working/test_satellite.png"

print("Running GeoChat inference on:", IMAGE_PATH)
answer = geochat_infer(IMAGE_PATH, QUESTION)

print()
print("=" * 60)
print("GEOCHAT ANSWER")
print("=" * 60)
print(answer)
print("=" * 60)

if not answer:
    raise RuntimeError("GeoChat returned an empty response.")

print()
print("✅ REAL GEOCHAT INFERENCE PASSED")


In [ ]:
# CELL 11 — FASTAPI
from fastapi import FastAPI, File, Form, UploadFile, HTTPException
from fastapi.responses import JSONResponse
from PIL import Image
from io import BytesIO
import time
import torch

app = FastAPI(title="SatQuery AI GeoChat API", version="1.0.0")

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": "MBZUAI/geochat-7B",
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }

@app.post("/v1/analyze")
async def analyze(
    image: UploadFile = File(...),
    question: str = Form(...),
):
    if not image.content_type or not image.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="Uploaded file must be an image")

    if not question.strip():
        raise HTTPException(status_code=400, detail="Question cannot be empty")

    image_bytes = await image.read()
    if len(image_bytes) > 20 * 1024 * 1024:
        raise HTTPException(status_code=413, detail="Image exceeds 20 MB limit")

    try:
        pil_image = Image.open(BytesIO(image_bytes)).convert("RGB")
    except Exception:
        raise HTTPException(status_code=400, detail="Invalid image format")

    start = time.time()
    try:
        ans = geochat_infer(pil_image, question)
    except Exception as e:
        print("GeoChat inference error:", repr(e))
        raise HTTPException(status_code=500, detail=f"GeoChat inference failed: {e}")

    elapsed = time.time() - start
    return JSONResponse({
        "success": True,
        "answer": ans,
        "model": "MBZUAI/geochat-7B",
        "inference_time": round(elapsed, 2),
    })

print("✅ FastAPI GeoChat API defined")


In [ ]:
# CELL 12 — START SERVER
import threading
import uvicorn
import time

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

print("🚀 GeoChat API running")
print("Local URL: http://127.0.0.1:8000")


In [ ]:
# CELL 13 — LOCAL HEALTH
import requests

response = requests.get("http://127.0.0.1:8000/health", timeout=10)
print("HTTP:", response.status_code)
print(response.json())

if response.status_code != 200:
    raise RuntimeError("GeoChat API health check failed")

print("✅ API HEALTH CHECK PASSED")


In [ ]:
# CELL 14 — LOCAL REAL INFERENCE
import requests
import os

with open(IMAGE_PATH, "rb") as f:
    response = requests.post(
        "http://127.0.0.1:8000/v1/analyze",
        files={"image": ("satellite.png", f, "image/png")},
        data={"question": "Describe what you see in this satellite image."},
        timeout=300,
    )

print("HTTP:", response.status_code)
result = response.json()
print(result)

if response.status_code != 200 or not result.get("answer"):
    raise RuntimeError(f"GeoChat API failed: {result}")

print()
print("✅ LOCAL API REAL INFERENCE PASSED")


In [ ]:
# CELL 15 — CLOUDFLARE
import subprocess
import os
import re
import time

CLOUDFLARED = "/kaggle/working/cloudflared"

if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O",
            CLOUDFLARED,
        ],
        check=True,
    )
    os.chmod(CLOUDFLARED, 0o755)

print("Starting cloudflared tunnel...")
tunnel = subprocess.Popen(
    [
        CLOUDFLARED,
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

PUBLIC_URL = None
for _ in range(60):
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(1)
        continue
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if match:
        PUBLIC_URL = match.group(0)
        print()
        print("=" * 60)
        print("PUBLIC GEOCHAT API")
        print("=" * 60)
        print(PUBLIC_URL)
        print("=" * 60)
        break

if PUBLIC_URL is None:
    raise RuntimeError("Cloudflare tunnel URL was not created")

print()
print("GEOCHAT_API_URL =", PUBLIC_URL)


In [ ]:
# CELL 16 — PUBLIC HEALTH
import requests

response = requests.get(f"{PUBLIC_URL}/health", timeout=30)
print("HTTP:", response.status_code)
print(response.json())

if response.status_code != 200:
    raise RuntimeError("Public GeoChat API is not reachable")

print()
print("✅ PUBLIC GEOCHAT API HEALTH PASSED")


In [ ]:
# CELL 17 — PUBLIC INFERENCE
import requests

with open(IMAGE_PATH, "rb") as f:
    response = requests.post(
        f"{PUBLIC_URL}/v1/analyze",
        files={"image": ("satellite.png", f, "image/png")},
        data={"question": "Describe what you see in this satellite image."},
        timeout=300,
    )

print("HTTP:", response.status_code)
result = response.json()
print(result)

if response.status_code != 200 or not result.get("answer"):
    raise RuntimeError("Public GeoChat API inference failed")

print()
print("✅ PUBLIC GEOCHAT API INFERENCE PASSED")
